# Production Data Analysis — Data Cleaning

This notebook focuses on cleaning and standardizing the raw production dataset identified during the initial data quality assessment.

In [4]:
from src.data_loading import load_production_data

df = load_production_data()

In [5]:

df.shape

(367, 16)

In [6]:
df_clean = df.copy()

In [7]:
df_clean = df_clean.drop_duplicates(
    subset=df_clean.columns.drop("production_id")
).copy()

In [8]:
df_clean.shape

(364, 16)

In [9]:
df.duplicated(
    subset=df.columns.drop("production_id")
).sum()

np.int64(3)

In [10]:
df[
    df.duplicated(
        subset=df.columns.drop("production_id"),
        keep=False
    )
].sort_values(
    by=df.columns.drop("production_id").tolist()
)

,production_id,date,shift,production_line,machine,product,operator_count,target_qty,actual_qty,good_qty,reject_qty,downtime_min,planned_time_min,cycle_time,defect_type,downtime_reason
99,100,2026-01-21,Morning,L2,M3,Product_C,7,978,1062,1052,10,31,480,0.42,NaN,Maintenance
362,514,2026-01-21,Morning,L2,M3,Product_C,7,978,1062,1052,10,31,480,0.42,NaN,Maintenance
19,20,2026-04-11,Morning,L2,M6,Product_B,6,1054,1062,1052,10,19,480,0.76,NaN,NaN
360,512,2026-04-11,Morning,L2,M6,Product_B,6,1054,1062,1052,10,19,480,0.76,NaN,NaN
249,250,2026-04-21,Night,L2,M3,Product_C,8,979,1062,1052,10,97,480,0.43,NaN,Setup/Changeover
365,517,2026-04-21,Night,L2,M3,Product_C,8,979,1062,1052,10,97,480,0.43,NaN,Setup/Changeover


In [11]:
#Shift Standardization
df_clean["shift"] = (
    df_clean["shift"]
    .str.strip()
    .str.title()
)

In [12]:
df_clean["shift"].value_counts()

shift
Evening    122
Morning    121
Night      121
Name: count, dtype: int64

In [13]:
df_clean["product"].value_counts(dropna=False)

product
Product_C      132
Product_B      119
Product_A      104
Product A        3
product_a        3
 Product_B       3
Name: count, dtype: int64

In [14]:
df_clean["product"] = (
    df_clean["product"]
    .str.strip()
    .str.replace(" ", "_", regex=False)
    .str.upper()
)

In [15]:
df_clean["product"].value_counts()

product
PRODUCT_C    132
PRODUCT_B    122
PRODUCT_A    110
Name: count, dtype: int64

In [17]:
import pandas as pd

In [18]:
df_clean["date"] = pd.to_datetime(df_clean["date"])

In [19]:
df_clean["date"].dtype

dtype('<M8[s]')

In [20]:
df_clean["date"].min(), df_clean["date"].max()

(Timestamp('2026-01-01 00:00:00'), Timestamp('2026-04-30 00:00:00'))

In [21]:
df_clean["defect_type"] = (
    df_clean["defect_type"]
    .fillna("No Defect")
)

In [22]:
df_clean["defect_type"].value_counts()

defect_type
No Defect          242
Dimension Error     31
Scratch             31
Color Defect        30
Surface Defect      30
Name: count, dtype: int64

In [23]:
#downtime_reason
df_clean[
    df_clean["downtime_reason"].isna()
]["downtime_min"].describe()

count     63.000000
mean      66.444444
std       31.358889
min       10.000000
25%       41.500000
50%       69.000000
75%       91.000000
max      120.000000
Name: downtime_min, dtype: float64

In [24]:
df_clean[
    df_clean["downtime_reason"].isna()
]["downtime_min"].value_counts()

downtime_min
69     3
26     3
57     3
101    2
120    2
12     2
100    2
44     2
63     2
95     2
81     2
62     2
80     1
29     1
36     1
86     1
19     1
71     1
10     1
110    1
113    1
78     1
114    1
72     1
50     1
115    1
11     1
45     1
85     1
84     1
31     1
39     1
119    1
77     1
37     1
43     1
33     1
117    1
70     1
51     1
27     1
64     1
65     1
90     1
92     1
97     1
40     1
74     1
Name: count, dtype: int64

In [25]:
df_clean.isnull().sum()

production_id        0
date                 0
shift                0
production_line      0
machine              0
product              0
operator_count       0
target_qty           0
actual_qty           0
good_qty             0
reject_qty           0
downtime_min         0
planned_time_min     0
cycle_time           0
defect_type          0
downtime_reason     63
dtype: int64

In [26]:
df_clean[
    (df_clean["target_qty"] < 0) |
    (df_clean["actual_qty"] < 0) |
    (df_clean["good_qty"] < 0) |
    (df_clean["reject_qty"] < 0) |
    (df_clean["downtime_min"] < 0) |
    (df_clean["planned_time_min"] < 0) |
    (df_clean["cycle_time"] <= 0)
]

,production_id,date,shift,production_line,machine,product,operator_count,target_qty,actual_qty,good_qty,reject_qty,downtime_min,planned_time_min,cycle_time,defect_type,downtime_reason


In [27]:
(df_clean["actual_qty"] == 
 df_clean["good_qty"] + df_clean["reject_qty"]).all()

np.True_

In [28]:
print("Shape:", df_clean.shape)
print("Duplicate rows:", df_clean.duplicated(
    subset=df_clean.columns.drop("production_id")
).sum())
print("Business rule valid:", (
    df_clean["actual_qty"] ==
    df_clean["good_qty"] + df_clean["reject_qty"]
).all())

Shape: (364, 16)
Duplicate rows: 0
Business rule valid: True


In [29]:
df_clean.to_csv(
    "E:\Robiul\Backup_my_own_laptop\Folder_1\Skills\End to End Data project\production-data-analysis\data\processed\production_data_clean.csv",
    index=False
)

<>:2: SyntaxWarning: invalid escape sequence '\R'
<>:2: SyntaxWarning: invalid escape sequence '\R'
C:\Users\PC\AppData\Local\Temp\ipykernel_15664\844054901.py:2: SyntaxWarning: invalid escape sequence '\R'
  "E:\Robiul\Backup_my_own_laptop\Folder_1\Skills\End to End Data project\production-data-analysis\data\processed\production_data_clean.csv",
